RAG Pipline - Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\hetmo\AppData\Local\Temp\ipykernel_11464\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


Data ingestion

In [2]:
###Read all the pdf's inside the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    #Find all pdf files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            #Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages from {pdf_file.name}")

        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

#Process all pdfs in the data directory
all_pdf_documents = process_all_pdfs("data")

Found 2 PDF files to process

Processing: BUS232 practice questions.pdf
Loaded 60 pages from BUS232 practice questions.pdf

Processing: final documentation.pdf
Loaded 7 pages from final documentation.pdf

Total documents loaded: 67


Chunking

In [3]:
### Text splitting into chunks

def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    """Split documents into smaller chunks for better RAG performance"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""] 
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    return split_docs

In [4]:
chunks = split_documents(all_pdf_documents)
chunks

Split 67 documents into 188 chunks


[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2020-05-13T21:59:29-07:00', 'author': 'Nathaniel Payne', 'moddate': '2020-05-13T21:59:29-07:00', 'source': 'data\\text_files\\pdf_files\\BUS232 practice questions.pdf', 'total_pages': 60, 'page': 0, 'page_label': '1', 'source_file': 'BUS232 practice questions.pdf', 'file_type': 'pdf'}, page_content='1 \n \nPractice Questions  \nSTAT 1123, 1124 & 1181 & BUS 232  \nTable Of Contents \nContents \nTable Of Contents ................................................................................................................................................................... 1 \nGraphs & Description (For Midterm 1) ................................................................................................................................... 1 \nDescriptive Statistics (For Midterm 1) .................................................................................................

Embedding and vectorStoreDB

In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\hetmo\OneDrive\Desktop\Het study\rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
class EmbeddingManager:
    """Handle embedding generation and storage using SentenceTransformers"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding manager
        
        Args: 
            model_name: HuggingFace model name for sentence embeddings
        
        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Loaded embedding model: {self.model_name}. Embedding dimension: {self.model.get_embedding_dimension()} ")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise e
        
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts
        
        Args:
            texts: List of strings to embed
        
        Returns:
            np.ndarray: Array of embeddings with shape (len(texts), embedding_dimension)
        """
        if not self.model:
            raise ValueError("Model not loaded. Call _load_model() first.")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, convert_to_numpy=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

### Initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 903.36it/s]


Loaded embedding model: all-MiniLM-L6-v2. Embedding dimension: 384 


Vector Store

In [17]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "data/vector_store"):
        """Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize the ChromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description": "Collection of PDF document embeddings for RAG"},
                configuration={"hnsw": {"space": "cosine"}},
                )
            print(f"Initialized ChromaDB collection: {self.collection_name} at {self.persist_directory}")
        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")
            raise e

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain document
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        "Prepare data for chromaDB"
        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = []

        for i, (doc, emb) in enumerate(zip(documents, embeddings)):
            #Generate a unique ID for each document
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{1}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)  
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(doc.metadata)

            documents_texts.append(doc.page_content)

            embeddings_list.append(emb.tolist())

        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas ,
                embeddings=embeddings_list,
                documents=documents_texts
            )
            print(f"Successfully added {len(documents)} documents to the vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise e

vector_store = VectorStore()
vector_store

Initialized ChromaDB collection: pdf_documents at data/vector_store


In [18]:
###Convert the text to embeddings

texts = [doc.page_content for doc in chunks]

#Generate embeddings for the text chunks
embeddings = embedding_manager.generate_embeddings(texts)

###Store in the vector database
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 188 texts...
Generated embeddings with shape: (188, 384)
Successfully added 188 documents to the vector store
Total documents in collection: 188


Retriever Pipeline from VectorStore

In [20]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store:VectorStore, embedding_manager: EmbeddingManager):
        """ Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Retrieve top_k relevant documents for a given query
        
        Args:
            query: User query string
            top_k: Number of top documents to retrieve
            score_threshold: Minimum similarity score for retrieved documents

        Returns:
            List of dictionaries containing document metadata and content
        """

        #Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    #Convert distance to similarity score (assuming distance is cosine distance)
                    similarity = 1 - (distance)
                    if similarity >= score_threshold:
                        doc_info = {
                            "similarity_score": float(similarity),
                            "metadata": metadata,
                            "content": document,
                            "id": doc_id,
                            "distance": float(distance),
                            "rank": i + 1

                        }
                        retrieved_docs.append(doc_info)
                print(f"Retrieved {len(retrieved_docs)} documents for query: '{query}'")

            else:
                print(f"No documents found for query: '{query}'")
        except Exception as e:
            print(f"Error occurred while querying the vector store: {e}")
            return []

        return retrieved_docs


rag_retriever = RAGRetriever(vector_store, embedding_manager)


In [21]:
rag_retriever.retrieve("What is issue with current art style?")

Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Retrieved 5 documents for query: 'What is issue with current art style?'


[{'similarity_score': 0.3504946231842041,
  'metadata': {'page': 1,
   'page_label': '2',
   'total_pages': 7,
   'file_type': 'pdf',
   'creator': 'Microsoft® Word LTSC',
   'producer': 'Microsoft® Word LTSC',
   'author': 'Het Modi',
   'moddate': '2023-12-11T15:20:27-08:00',
   'source': 'data\\text_files\\pdf_files\\final documentation.pdf',
   'source_file': 'final documentation.pdf',
   'creationdate': '2023-12-11T15:20:27-08:00'},
  'content': "adaptability, they can be used to solve issues whose solution area are ill-defined and require \ninvestigation before reaching an aesthetic interpretation. \nTarget audience \nThis project is intended for designers, artists, and enthusiasts who want to investigate the \ncombination of contemporary generative approaches with classic artistic expression. It also \nfunctions as a teaching tool for comprehending the fundamentals of genetic algorithms and how \nto use them in artistic projects. \nChallenges and opportunities \nThe challenge li